# The best-fit line and its loss bowl
**Machine Learning · Week 2 (Regression and gradient descent) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w2d1.html

How much should a pizza cost, given its diameter? We fit a straight line
$\hat{y} = w x + b$ by making the **mean squared error** (MSE) as small as possible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
diameter = rng.uniform(20, 40, size=25)                            # cm
price = 0.55 * diameter - 2 + rng.normal(0, 1.4, size=25)          # dollars, with some noise

plt.scatter(diameter, price)
plt.xlabel("pizza diameter (cm)")
plt.ylabel("price ($)")
plt.title("25 pizzas")
plt.show()

## Scoring a line
A **residual** is the vertical gap between a data point and the line. Square every residual and
average them: that's the MSE. Smaller is better.

In [ ]:
def predict(x, w, b):
    return w * x + b

def mse(x, y, w, b):
    residuals = y - predict(x, w, b)
    return np.mean(residuals ** 2)

for w, b in [(0.3, 5.0), (0.6, -3.0), (0.5, -0.5)]:
    print(f"w = {w:4.2f}, b = {b:5.2f}  ->  MSE = {mse(diameter, price, w, b):6.2f}")

## The best line
For a straight line there is an exact formula (**least squares**). `np.polyfit` applies it.

In [ ]:
w_best, b_best = np.polyfit(diameter, price, deg=1)
print(f"best line: price = {w_best:.3f} * diameter {b_best:+.3f}")
print(f"its MSE  : {mse(diameter, price, w_best, b_best):.3f}")

In [ ]:
xs = np.linspace(18, 42, 100)
fig, ax = plt.subplots(layout="constrained")
for x, y in zip(diameter, price):
    ax.plot([x, x], [y, predict(x, w_best, b_best)], color="#ff8a50", lw=1.2, alpha=0.8)  # residuals
ax.scatter(diameter, price, zorder=3)
ax.plot(xs, predict(xs, w_best, b_best), color="#7b5cfa", lw=2.5, label="least-squares line")
ax.set_xlabel("pizza diameter (cm)")
ax.set_ylabel("price ($)")
ax.legend()
plt.show()

## The scikit-learn way
scikit-learn expects the features as a 2-D table (rows = examples, columns = features),
hence the `reshape(-1, 1)`.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(diameter.reshape(-1, 1), price)
print("slope w    :", model.coef_[0].round(3))
print("intercept b:", model.intercept_.round(3))
print("a 32 cm pizza should cost about $", model.predict([[32]])[0].round(2))

## The loss bowl
Every pair $(w, b)$ is one line, and every line has an MSE. Drawing the MSE over all pairs gives a
landscape: a long, narrow **bowl**. The best line sits at its bottom.

In [ ]:
ws = np.linspace(0.0, 1.1, 221)
bs = np.linspace(-16, 12, 281)
W, B = np.meshgrid(ws, bs)                                    # every (w, b) pair on a grid
# broadcasting: predictions for every grid point and every pizza at once
pred = W[..., None] * diameter + B[..., None]                 # shape (281, 221, 25)
loss = np.mean((price - pred) ** 2, axis=-1)                  # one MSE per grid point

fig, ax = plt.subplots(figsize=(6.2, 4.2), layout="constrained")
levels = np.geomspace(loss.min(), loss.max(), 18)
cs = ax.contourf(W, B, loss, levels=levels, cmap="Purples_r", norm="log")
ax.contour(W, B, loss, levels=levels, colors="white", linewidths=0.5, alpha=0.6)
ax.plot(w_best, b_best, "*", ms=16, color="#ffd66b", mec="#2c2842", label="least-squares line")
ax.set_xlabel("slope w")
ax.set_ylabel("intercept b")
ax.set_title("MSE for every possible line (darker = smaller)")
ax.legend(loc="upper right")
plt.show()

## Your turn: find the bottom by brute force
`loss` holds the MSE of 62,000 lines. Find the grid point with the smallest MSE, then compare
it with `np.polyfit`'s answer.

In [ ]:
i = 0   # TODO: the flat index of the smallest loss: loss.argmin()
row, col = np.unravel_index(i, loss.shape)
print(f"grid best:  w = {W[row, col]:.3f}, b = {B[row, col]:.2f}, MSE = {loss[row, col]:.3f}")
print(f"polyfit:    w = {w_best:.3f}, b = {b_best:.2f}, MSE = {mse(diameter, price, w_best, b_best):.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
i = loss.argmin()
row, col = np.unravel_index(i, loss.shape)
print(f"grid best:  w = {W[row, col]:.3f}, b = {B[row, col]:.2f}, MSE = {loss[row, col]:.3f}")
print(f"polyfit:    w = {w_best:.3f}, b = {b_best:.2f}, MSE = {mse(diameter, price, w_best, b_best):.3f}")
print("Brute force needs 62,000 tries for two knobs. With a million knobs a grid is hopeless:")
print("that's why tomorrow's lesson walks downhill instead.")